# Xử lý passage IELTS Reading từ tiếng anh và phần dịch tiếng việt trong file .txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import zipfile
import re
from collections import Counter

In [ ]:
zip_path = "/content/drive/MyDrive/academic_sentence.zip"
extract_dir = "/content/ielts_raw"

In [ ]:
# Giải nén
import shutil
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

In [ ]:
# Giải nén và thu thập file có dạng .txt
all_txt = []
for root, dirs, files in os.walk(extract_dir):

    for f in files:
        if f.endswith(".txt"):
            all_txt.append(os.path.join(root, f))

In [ ]:
# Phân loại.
en_files = sorted([f for f in all_txt if "_en.txt" in os.path.basename(f)])
vi_files = sorted([f for f in all_txt if "_vi.txt" in os.path.basename(f)])

In [ ]:
# Ghép cặp theo prefix.
def get_prefix(path):
    name = os.path.basename(path)
    return name.split("_")[0]

en_dict = {get_prefix(f): f for f in en_files}
vi_dict = {get_prefix(f): f for f in vi_files}
common_keys = sorted(set(en_dict.keys()) & set(vi_dict.keys()))

In [ ]:
print(f"Tổng file .txt tìm thấy: {len(all_txt)}")
print(f"File EN: {len(en_files)}")
print(f"File VI: {len(vi_files)}")
print(f"Cặp ghép được: {len(common_keys)}")

Tổng file .txt tìm thấy: 336
File EN: 168
File VI: 168
Cặp ghép được: 168


In [ ]:
# Đọc file.
def read_file(path):
    for enc in ["utf-8", "utf-8-sig", "latin-1"]:
        try:
            with open(path, "r", encoding=enc) as f:
                return f.read()
        except UnicodeDecodeError:
            continue
    return ""

In [ ]:
# Đọc và ghép nội dung dưới dạng từ điển
data = []
for key in common_keys:
    en_text = read_file(en_dict[key]).strip()
    vi_text = read_file(vi_dict[key]).strip()
    data.append({"id": key, "en": en_text, "vi": vi_text})

In [ ]:
# Tính độ dài đoạn văn mỗi file
en_lens = [len(d["en"]) for d in data]
vi_lens = [len(d["vi"]) for d in data]

In [ ]:
print(f"Số cặp: {len(data)}")
print(f"Độ dài EN (ký tự) - min: {min(en_lens)}, max: {max(en_lens)}, avg: {sum(en_lens)//len(en_lens)}")
print(f"Độ dài VI (ký tự) - min: {min(vi_lens)}, max: {max(vi_lens)}, avg: {sum(vi_lens)//len(vi_lens)}")

Số cặp: 168
Độ dài EN (ký tự) - min: 3787, max: 7275, avg: 5317
Độ dài VI (ký tự) - min: 4028, max: 7094, avg: 5607


In [ ]:
# Hàm để kiểm tra các lỗi dấu câu.
def check_punctuation(text, lang):
    issues = []

    # Không kết thúc bằng dấu câu.
    if text and text[-1] not in ".?!\"'»)":
        issues.append(f"Không kết thúc bằng dấu câu, ký tự cuối: '{text[-1]}'")

    # Nháy đơn/kép không đóng.
    if text.count('"') % 2 != 0:
        issues.append(f'Nháy kép (") lẻ: {text.count(chr(34))} cái')
    if text.count("'") % 2 != 0:
        # Tiếng Anh hay dùng apostrophe
        if lang == "vi":
            issues.append(f"Nháy đơn (') lẻ: {text.count(chr(39))} cái")

    # Smart quotes
    left_dq = text.count("\u201c")  # "
    right_dq = text.count("\u201d")  # "
    if left_dq != right_dq:
        issues.append(f"Smart quotes kép lệch: \u201c={left_dq}, \u201d={right_dq}")

    left_sq = text.count("\u2018")  # '
    right_sq = text.count("\u2019")  # '
    if lang == "vi" and left_sq != right_sq:
        issues.append(f"Smart quotes đơn lệch: \u2018={left_sq}, \u2019={right_sq}")

    # Dấu chấm kép, ba chấm bất thường.
    double_dots = len(re.findall(r'(?<!\.)\.\.(?!\.)', text))
    if double_dots:
        issues.append(f"Dấu chấm kép (..) không phải ba chấm: {double_dots} chỗ")

    # Khoảng trắng trước dấu câu.
    space_before = len(re.findall(r'\s[.,:;!?]', text))
    if space_before:
        issues.append(f"Khoảng trắng trước dấu câu: {space_before} chỗ")

    # Thiếu khoảng trắng sau dấu câu (nối liền chữ).
    no_space_after = len(re.findall(r'[.!?][A-Za-zÀ-ỹ]', text))
    if no_space_after:
        issues.append(f"Thiếu space sau dấu câu: {no_space_after} chỗ")

    # Dấu phẩy cuối câu thay vì dấu chấm.
    comma_end = len(re.findall(r',\s*$', text, re.MULTILINE))
    if comma_end:
        issues.append(f"Dòng kết thúc bằng dấu phẩy: {comma_end}")

    return issues

In [ ]:
en_issue_count = 0
vi_issue_count = 0

In [ ]:
for d in data:
    en_issues = check_punctuation(d["en"], "en")
    vi_issues = check_punctuation(d["vi"], "vi")

    if en_issues or vi_issues:
        print(f"\n--- {d['id']} ---")
        if en_issues:
            en_issue_count += 1
            for iss in en_issues:
                print(f"  EN: {iss}")
        if vi_issues:
            vi_issue_count += 1
            for iss in vi_issues:
                print(f"  VI: {iss}")

print(f"Tổng kết: {en_issue_count}/{len(data)} file EN có lỗi, {vi_issue_count}/{len(data)} file VI có lỗi")


--- 003 ---
  EN: Thiếu space sau dấu câu: 1 chỗ

--- 004 ---
  EN: Không kết thúc bằng dấu câu, ký tự cuối: '’'
  VI: Nháy đơn (') lẻ: 11 cái
  VI: Smart quotes đơn lệch: ‘=0, ’=1

--- 005 ---
  EN: Thiếu space sau dấu câu: 2 chỗ

--- 006 ---
  VI: Nháy đơn (') lẻ: 7 cái

--- 007 ---
  VI: Nháy đơn (') lẻ: 7 cái

--- 008 ---
  VI: Khoảng trắng trước dấu câu: 1 chỗ

--- 010 ---
  VI: Smart quotes kép lệch: “=0, ”=2
  VI: Smart quotes đơn lệch: ‘=1, ’=2

--- 015 ---
  EN: Khoảng trắng trước dấu câu: 3 chỗ
  VI: Khoảng trắng trước dấu câu: 3 chỗ

--- 019 ---
  VI: Smart quotes kép lệch: “=3, ”=1
  VI: Smart quotes đơn lệch: ‘=1, ’=2

--- 020 ---
  VI: Thiếu space sau dấu câu: 1 chỗ

--- 021 ---
  EN: Thiếu space sau dấu câu: 1 chỗ

--- 025 ---
  EN: Thiếu space sau dấu câu: 1 chỗ

--- 026 ---
  VI: Khoảng trắng trước dấu câu: 1 chỗ

--- 027 ---
  VI: Nháy đơn (') lẻ: 5 cái

--- 030 ---
  EN: Smart quotes kép lệch: “=1, ”=0
  EN: Khoảng trắng trước dấu câu: 1 chỗ
  EN: Thiếu space sau dấ

In [ ]:
# Tách câu theo dấu kết thúc. Tách câu đơn giản chỉ để thống kê.
def split_sentences(text):
    text = re.sub(r'\b(Mr|Mrs|Ms|Dr|Prof|e\\.g|i\\.e|etc|vs|et al|Fig|Ref|Vol|Eq|Ph\\.D|Inc|Ltd|Co|Corp|St)\\.',
                  lambda m: m.group(0).replace('.', '<DOT>'), text)

    sents = re.split(r'(?<=[.!?])\s+(?=[A-ZÀ-Ỹ"\'\u201c\u2018(])', text)

    sents = [s.replace('<DOT>', '.').strip() for s in sents if s.strip()]
    return sents

alignment_data = []
for d in data:
    en_sents = split_sentences(d["en"])
    vi_sents = split_sentences(d["vi"])
    diff = len(en_sents) - len(vi_sents)
    alignment_data.append({
        "id": d["id"],
        "en_count": len(en_sents),
        "vi_count": len(vi_sents),
        "diff": diff,
        "abs_diff": abs(diff),
        "en_sents": en_sents,
        "vi_sents": vi_sents,
        })

In [ ]:
# Thống kê phân loại theo mức độ lệch.
diffs = [a["abs_diff"] for a in alignment_data]
diff_counter = Counter(diffs)


print(f"\nTổng cặp: {len(alignment_data)}")
print(f"Khớp hoàn toàn (diff=0): {diff_counter[0]})")
print(f"Lệch 1-2 câu: {sum(diff_counter[k] for k in diff_counter if 1 <= k <= 2)}")
print(f"Lệch 3-5 câu: {sum(diff_counter[k] for k in diff_counter if 3 <= k <= 5)}")
print(f"Lệch 6+ câu: {sum(diff_counter[k] for k in diff_counter if k >= 6)}")

print(f"Phân bố chi tiết:")
for diff_val in sorted(diff_counter.keys()):
    bar = "█" * diff_counter[diff_val]
    print(f"  Lệch {diff_val:>2} câu: {diff_counter[diff_val]:>3} cặp  {bar}")


Tổng cặp: 168
Khớp hoàn toàn (diff=0): 119)
Lệch 1-2 câu: 33
Lệch 3-5 câu: 15
Lệch 6+ câu: 1
Phân bố chi tiết:
  Lệch  0 câu: 119 cặp  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  Lệch  1 câu:  26 cặp  ██████████████████████████
  Lệch  2 câu:   7 cặp  ███████
  Lệch  3 câu:   9 cặp  █████████
  Lệch  4 câu:   2 cặp  ██
  Lệch  5 câu:   4 cặp  ████
  Lệch  8 câu:   1 cặp  █


In [ ]:
# In ra thông tin của 10 cặp có độ lệch lớn nhất để kiểm tra.
sorted_by_diff = sorted(alignment_data, key=lambda x: x["abs_diff"], reverse=True)

for a in sorted_by_diff[:10]:
    print(f"{a['id']} | EN: {a['en_count']} câu, VI: {a['vi_count']} câu, lệch: {a['abs_diff']} ---")
    print(f"EN đầu: {a['en_sents'][0][:100]}...")
    print(f"VI đầu: {a['vi_sents'][0][:100]}...")
    print(f"EN cuối: {a['en_sents'][-1][:100]}...")
    print(f"VI cuối: {a['vi_sents'][-1][:100]}...")

127 | EN: 44 câu, VI: 36 câu, lệch: 8 ---
EN đầu: Scientific discovery is popularly believed to result from the sheer genius of such intellectual star...
VI đầu: Khám phá khoa học được nhiều người cho là kết quả từ thiên tài tuyệt đối của những ngôi sao trí tuệ ...
EN cuối: The time seems right for abandoning the naive notions of intelligent design and genius, and for scie...
VI cuối: Có vẻ như đây là thời điểm thích hợp để từ bỏ những quan niệm ngây thơ về thiết kế thông minh và thi...
021 | EN: 39 câu, VI: 44 câu, lệch: 5 ---
EN đầu: Our life span is restricted....
VI đầu: Tuổi thọ của chúng ta bị hạn chế....
EN cuối: This final aspect should not be forgotten....
VI cuối: Khía cạnh cuối cùng này không nên bị lãng quên....
045 | EN: 36 câu, VI: 31 câu, lệch: 5 ---
EN đầu: One of the most captivating natural events of the year in many areas throughout North America is the...
VI đầu: Một trong những sự kiện tự nhiên hấp dẫn nhất trong năm ở nhiều khu vực trên khắp Bắc Mỹ là sự thay ...


In [ ]:
small_diff = [a for a in alignment_data if 1 <= a["abs_diff"] <= 2]

print(f"Lệch nhỏ 1-2 câu ({len(small_diff)} cặp)")


for a in small_diff[:2]:
    d = a["abs_diff"]
    tail = d + 3  # số câu cuối cần in để thấy vùng lệch
    en_tail = a["en_sents"][-tail:]
    vi_tail = a["vi_sents"][-tail:]
    en_start = a["en_count"] - len(en_tail) + 1
    vi_start = a["vi_count"] - len(vi_tail) + 1

    print(f"FILE: {a['id']} | EN: {a['en_count']} câu, VI: {a['vi_count']} câu, lệch: {d}")
    print(f"  EN (câu {en_start}-{a['en_count']}):")
    for i, s in enumerate(en_tail, en_start):
        print(f"[{i:>3}] {s}")
    print(f"  VI (câu {vi_start}-{a['vi_count']}):")
    for i, s in enumerate(vi_tail, vi_start):
        print(f"[{i:>3}] {s}")

Lệch nhỏ 1-2 câu (33 cặp)
FILE: 009 | EN: 50 câu, VI: 51 câu, lệch: 1
  EN (câu 47-50):
[ 47] But this is the result of dedicated work over a long period, particularly from the officers in charge of community development.
[ 48] They played an essential role in raising awareness and interest among the rural communities.
[ 49] The concept of integrated rural transport is now well established in Tanzania, where a major program of rural transport is just about to start.
[ 50] The experiences from Makete will help in this initiative, and Makete District will act as a reference for future work.
  VI (câu 48-51):
[ 48] Nhưng đây là kết quả của sự làm việc tận tâm trong một thời gian dài, đặc biệt là từ các cán bộ phụ trách phát triển cộng đồng.
[ 49] Họ đóng một vai trò thiết yếu trong việc nâng cao nhận thức và sự quan tâm giữa các cộng đồng nông thôn.
[ 50] Khái niệm về giao thông nông thôn tích hợp hiện đã được thiết lập tốt ở Tanzania, nơi một chương trình lớn về giao thông nông thôn sắp 

In [ ]:
big_diff = [a for a in alignment_data if a["abs_diff"] >= 6]

print(f"Ví dụ lệch 6+ câu ({len(big_diff)} cặp) - BertAlign")

for a in big_diff[:2]:
    d = a["abs_diff"]
    tail = d + 3
    en_tail = a["en_sents"][-tail:]
    vi_tail = a["vi_sents"][-tail:]
    en_start = a["en_count"] - len(en_tail) + 1
    vi_start = a["vi_count"] - len(vi_tail) + 1


    print(f"File: {a['id']} | EN: {a['en_count']} câu, VI: {a['vi_count']} câu, lệch: {d}")
    print(f"EN (câu {en_start}-{a['en_count']}):")
    for i, s in enumerate(en_tail, en_start):
        print(f"[{i:>3}] {s}")
    print(f"VI (câu {vi_start}-{a['vi_count']}):")
    for i, s in enumerate(vi_tail, vi_start):
        print(f"[{i:>3}] {s}")

Ví dụ lệch 6+ câu (1 cặp) - BertAlign
File: 127 | EN: 44 câu, VI: 36 câu, lệch: 8
EN (câu 34-44):
[ 34] Far more banal and mechanical forces may be at work; forces that are fundamentally connected to the laws of science.
[ 35] The notions of insight, creativity and genius are often invoked, but they remain vague and of doubtful scientific utility, especially when one considers the diverse and enduring contributions of individuals such as Plato, Leonardo da Vinci, Shakespeare, Beethoven, Galileo, Newton, Kepler, Curie, Pasteur and Edison.
[ 36] These notions merely label rather than explain the evolution of human innovations.
[ 37] We need another approach, and there is a promising candidate.
[ 38] The Law of Effect was advanced by psychologist Edward Thorndike in 1898, some 40 years after Charles Darwin published his groundbreaking work on biological evolution, On the Origin of Species.
[ 39] This simple law holds that organisms tend to repeat successful behaviors and to refrain from p

In [ ]:
# Phân bố các mức lệch
total = len(alignment_data)
perfect = diff_counter[0]
gale_church = sum(diff_counter[k] for k in diff_counter if 1 <= k <= 2)
medium = sum(diff_counter[k] for k in diff_counter if 3 <= k <= 5)
bertalign = sum(diff_counter[k] for k in diff_counter if k >= 6)

print("Tổng kết")

print(f"""Tổng cặp file:     {total}
Khớp (diff=0):     {perfect}  ({100*perfect//total}%)
Lệch 1-2 câu:      {gale_church}  ({100*gale_church//total}%)
Lệch 3-5 câu:      {medium}  ({100*medium//total}%)
Lệch 6+ câu:       {bertalign}  ({100*bertalign//total}%)

Tổng câu EN:        {sum(a['en_count'] for a in alignment_data)}
Tổng câu VI:        {sum(a['vi_count'] for a in alignment_data)}
""")

Tổng kết
Tổng cặp file:     168
Khớp (diff=0):     119  (70%)
Lệch 1-2 câu:      33  (19%)
Lệch 3-5 câu:      15  (8%)
Lệch 6+ câu:       1  (0%)

Tổng câu EN:        6405
Tổng câu VI:        6396



### Căn chỉnh ngữ nghĩa

In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/LaBSE", device=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
def gale_church_align(en_sents, vi_sents):
    """
    Căn chỉnh câu dựa trên độ dài (Gale-Church Algorithm / Length-based DP)
    Phù hợp cho các văn bản có độ lệch nhỏ (1-2 câu).
    """
    n, m = len(en_sents), len(vi_sents)
    if n == 0 or m == 0: return []

    len_en = [len(s) for s in en_sents]
    len_vi = [len(s) for s in vi_sents]

    # Tính tỷ lệ ký tự trung bình giữa VI và EN
    c = sum(len_vi) / max(sum(len_en), 1)

    # Hàm tính phạt chi phí nếu lệch nặng.
    def match_cost(l_en, l_vi):
        expected = l_en * c
        if expected == 0 and l_vi == 0: return 0

        # So sánh độ dài thực tế với độ dài kỳ vọng
        ratio = min(expected, l_vi) / max(expected, l_vi)
        return (1.0 - ratio) * 10

    # Khởi tạo ma trận Quy hoạch động
    D = np.full((n + 1, m + 1), np.inf)
    D[0, 0] = 0
    backtrack = {}

    # Các bước ghép (di, dj, penalty): 1-1, 1-2, 2-1, 2-2, 3-1, 1-3
    operations = [
        (1, 1, 0),    # Ghép 1-1 (Ưu tiên nhất)
        (2, 1, 2.5),  # Ghép 2 EN - 1 VI
        (1, 2, 2.5),  # Ghép 1 EN - 2 VI
        (2, 2, 3.5),  # Ghép 2-2
        (3, 1, 5.0),  # Ghép 3 EN - 1 VI
        (1, 3, 5.0)]  # Ghép 1 EN - 3 VI


    # Thuật toán tìm đường đi ngắn nhất (Viterbi/DP)
    for i in range(n + 1):
        for j in range(m + 1):
            if D[i, j] == np.inf: continue

            for di, dj, penalty in operations:
                ni, nj = i + di, j + dj
                if ni > n or nj > m: continue

                l_en = sum(len_en[i:ni])
                l_vi = sum(len_vi[j:nj])

                cost = match_cost(l_en, l_vi) + penalty

                if D[i, j] + cost < D[ni, nj]:
                    D[ni, nj] = D[i, j] + cost
                    backtrack[(ni, nj)] = (i, j, di, dj)

    # Truy vết ngược để lấy kết quả
    aligned_pairs = []
    i, j = n, m
    while i > 0 or j > 0:
        if (i, j) not in backtrack: break
        prev_i, prev_j, di, dj = backtrack[(i, j)]

        en_text = " ".join(en_sents[prev_i:i]).strip()
        vi_text = " ".join(vi_sents[prev_j:j]).strip()

        length_ratio = min(len(en_text), max(1, len(vi_text))) / max(len(en_text), max(1, len(vi_text)))
        # Đặt sim_score = None thì rỗng ở dataframe
        aligned_pairs.append((en_text, vi_text, round(length_ratio, 3), f"{di}-{dj}", np.nan))

        i, j = prev_i, prev_j

    return aligned_pairs[::-1]

In [ ]:
# Sử sử dụng mô hình LaBSE để thực hiện căn chỉnh ngữ nghĩa giữa các đoạn lệch nặng
# Tính toán ma trận tương đồng và thuật toán đường đi ngắn nhất để tìm các cặp câu tốt nhất.
def semantic_align(en_sents, vi_sents, model):
    n, m = len(en_sents), len(vi_sents)
    if n == 0 or m == 0: return []

    en_texts, vi_texts = [], []
    en_idx_map, vi_idx_map = {}, {}

    for i in range(n):
        for di in range(1, 4):
            if i + di <= n:
                en_idx_map[(i, di)] = len(en_texts)
                en_texts.append(" ".join(en_sents[i:i+di]).strip())

    for j in range(m):
        for dj in range(1, 4):
            if j + dj <= m:
                vi_idx_map[(j, dj)] = len(vi_texts)
                vi_texts.append(" ".join(vi_sents[j:j+dj]).strip())

    # Encode bằng GPU
    en_embs = model.encode(en_texts, convert_to_tensor=True, show_progress_bar=False)
    vi_embs = model.encode(vi_texts, convert_to_tensor=True, show_progress_bar=False)

    # Tính ma trận tương đồng
    sim_matrix = util.cos_sim(en_embs, vi_embs)

    D = np.full((n + 1, m + 1), np.inf)
    D[0, 0] = 0
    backtrack = {}

    operations = [
        (1, 1, 0.0),
        (2, 1, 0.1), (1, 2, 0.1),
        (3, 1, 0.2), (1, 3, 0.2),
        (2, 2, 0.3),
        (1, 0, 0.7), (0, 1, 0.7)]

    for i in range(n + 1):
        for j in range(m + 1):
            if D[i, j] == np.inf: continue

            for di, dj, penalty in operations:
                ni, nj = i + di, j + dj
                if ni > n or nj > m: continue

                sim_val = 0.0
                if di > 0 and dj > 0:
                    idx_en = en_idx_map[(i, di)]
                    idx_vi = vi_idx_map[(j, dj)]
                    sim_val = sim_matrix[idx_en][idx_vi].item()

                    # Logic phạt similarity thấp
                    step_cost = (1.0 - sim_val) + penalty
                    if sim_val < 0.30: step_cost += 2.0
                    if sim_val < 0.20: step_cost += 5.0
                else:
                    step_cost = penalty

                if D[i, j] + step_cost < D[ni, nj]:
                    D[ni, nj] = D[i, j] + step_cost
                    backtrack[(ni, nj)] = (i, j, di, dj, sim_val)

    aligned_pairs = []
    i, j = n, m
    while i > 0 or j > 0:
        if (i, j) not in backtrack: break
        prev_i, prev_j, di, dj, sim_val = backtrack[(i, j)]
        en_text = " ".join(en_sents[prev_i:i]).strip()
        vi_text = " ".join(vi_sents[prev_j:j]).strip()

        if en_text and vi_text:
            length_ratio = min(len(en_text), len(vi_text)) / max(len(en_text), len(vi_text))
            aligned_pairs.append((en_text, vi_text, round(length_ratio, 3), f"{di}-{dj}", round(sim_val, 4)))
        i, j = prev_i, prev_j

    return aligned_pairs[::-1]

In [ ]:
final_data = []
for idx, a in enumerate(alignment_data):
    file_id = a["id"]
    en_sents = a["en_sents"]
    vi_sents = a["vi_sents"]
    abs_diff = a["abs_diff"]

    # Khớp (70% dữ liệu) thì ghép 1-1 trực tiếp
    if abs_diff == 0:
        for i in range(len(en_sents)):
            en_text = en_sents[i].strip()
            vi_text = vi_sents[i].strip()

            if not en_text or not vi_text:
                continue

            length_ratio = min(len(en_text), len(vi_text)) / max(len(en_text), len(vi_text))
            final_data.append({
                "file_id": file_id,
                "en": en_text,
                "vi": vi_text,
                "length_ratio": round(length_ratio, 3),
                "alignment_type": "1-1",
                "semantic_score": 1.0,
                "method": "Direct"   })


    # Lệch nhẹ (1-2 câu) thì Gale-Church xử lý
    elif 1 <= abs_diff <= 2:
        aligned_pairs = gale_church_align(en_sents, vi_sents)
        for pair in aligned_pairs:
            en_text, vi_text, length_ratio, align_type, _ = pair
            final_data.append({
                "file_id": file_id,
                "en": en_text,
                "vi": vi_text,
                "length_ratio": length_ratio,
                "alignment_type": align_type,
                "semantic_score": np.nan, # None không hiển thị trên dataframe
                "method": "Gale-Church"})


    # Bị lệch câu nặng (>= 3 câu) thì LaBSE xử lý
    else:
        aligned_pairs = semantic_align(en_sents, vi_sents, model)
        for pair in aligned_pairs:
            en_text, vi_text, length_ratio, align_type, sim_score = pair
            final_data.append({
                "file_id": file_id,
                "en": en_text,
                "vi": vi_text,
                "length_ratio": length_ratio,
                "alignment_type": align_type,
                "semantic_score": sim_score,
                "method": "LaBSE"
            })

print(f"Tổng số cặp câu thu được: {len(final_data)}")

Tổng số cặp câu thu được: 6304


In [ ]:
# Lưu data
df_final = pd.DataFrame(final_data)
df_final.replace("", np.nan, inplace=True)
df_final.dropna(subset=["en", "vi"], inplace=True)
df_final.reset_index(drop=True, inplace=True)

# Xuất Parquet
output_path = "/content/ielts_academic_clean.parquet"
df_final.to_parquet(output_path, engine="pyarrow", index=False)
print(f"Đã lưu Parquet tại: {output_path}")

Đã lưu Parquet tại: /content/ielts_academic_clean.parquet


In [ ]:
# Xác định số lượng câu theo từng phương pháp
direct_count = len(df_final[df_final["method"] == "Direct"])
gc_count = len(df_final[df_final["method"] == "Gale-Church"])
labse_count = len(df_final[df_final["method"] == "LaBSE"])

print(f"Thu được {len(df_final)} cặp câu song ngữ")

print(f"1. Ghép trực tiếp 1-1: {direct_count} cặp")
print(f"2. Thuật toán Gale-Church: {gc_count} cặp")
print(f"3. LaBSE: {labse_count} cặp")

print(df_final["alignment_type"].value_counts().to_string())

labse_scores = df_final[df_final["method"] == "LaBSE"]["semantic_score"]
if len(labse_scores) > 0:
    print(labse_scores.describe().to_string())

# Lọc trực tiếp trên nhóm LaBSE để lấy các cặp có điểm cực thấp
bad_pairs = df_final[(df_final["method"] == "LaBSE") & (df_final["semantic_score"] < 0.40)]
print(f"Số lượng cặp 'Bad Pairs': {len(bad_pairs)}")

if len(bad_pairs) > 0:
    print("Bad Pairs:")
    for _, row in bad_pairs.iterrows():
        print(f"\n[File: {row['file_id']} | Score: {row['semantic_score']}]")
        print(f"EN: {row['en']}")
        print(f"VI: {row['vi']}")

Thu được 6304 cặp câu song ngữ
1. Ghép trực tiếp 1-1: 4539 cặp
2. Thuật toán Gale-Church: 1209 cặp
3. LaBSE: 556 cặp
alignment_type
1-1    6179
1-2      34
2-2      28
3-1      25
2-1      23
1-3      15
count    556.000000
mean       0.824788
std        0.120271
min        0.305100
25%        0.821250
50%        0.859400
75%        0.886325
max        0.942600
Số lượng cặp 'Bad Pairs': 15
Bad Pairs:

[File: 021 | Score: 0.3586]
EN: Our life span is restricted. Everyone accepts this as ‘biologically’ obvious. ‘Nothing lives forever!’ However, in this statement we think of artificially produced, technical objects, products which are subjected to natural wear and tear during use.
VI: Do đó, lão hóa và cái chết không nên được coi là không thể tránh khỏi, đặc biệt là khi sinh vật sở hữu nhiều bộ phận để sửa chữa.

[File: 041 | Score: 0.3776]
EN: Alan Macfarlane, professor of anthropological science at King’s College, Cambridge has, like other historians, spent decades wrestling with the en

### Download Cleaned Data

In [ ]:
import zipfile
import os
from google.colab import files


zip_file_name = "ielts_academic_clean.zip"

# Create a ZipFile object and add the parquet file to it
with zipfile.ZipFile(zip_file_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(output_path, os.path.basename(output_path))

print(f"File '{output_path}' has been zipped to '{zip_file_name}'")

# Offer the zip file for download
files.download(zip_file_name)

File '/content/ielts_academic_clean.parquet' has been zipped to 'ielts_academic_clean.zip'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>